# 06 — Productionization & Deployment

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 6 — Productionization & Deployment

## Objective

Turn the Phase 4B XGBoost development candidate into a reproducible inference
service without changing the trained model.

The serving API returns a **risk score**, not a calibrated probability.

Phase 5 showed:

- useful but imperfect future ranking;
- poor probability calibration;
- temporal threshold instability;
- significant drift in `promised_delivery_days`.

Therefore the production contract emphasizes ranking, explicit model metadata,
configurable operational policy, logging and monitoring.

## 1. Serving architecture

```text
Order approved
      ↓
upstream feature computation
      ↓
41 safe Phase 3 features
      ↓
Pydantic validation
      ↓
XGBoost sklearn Pipeline
      ↓
risk_score
      ↓
OOF-derived risk band
      ↓
configurable review policy
```

The API does not calculate raw Olist joins online.

## 2. Install FastAPI

In [ ]:
# Run once:
#
# uv add "fastapi[standard]"

## 3. Build deployment configuration

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()

for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "models").exists():
        PROJECT_ROOT = candidate
        break

print("PROJECT_ROOT:", PROJECT_ROOT)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "src.serving.build_deployment_config",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "Deployment config build failed."
    )

## 4. Inspect deployment metadata

In [ ]:
import json

with (
    PROJECT_ROOT
    / "models"
    / "deployment_config.json"
).open(
    "r",
    encoding="utf-8",
) as file:
    deployment_config = json.load(file)

deployment_config

## 5. Important semantics

The response field is:

`risk_score`

not:

`late_delivery_probability`

Risk bands are based on development temporal-OOF score quantiles.

The `requires_review` threshold is an operational configuration selected from
development OOF. It does not guarantee the same recall in future periods.

## 6. Build an API request from an existing feature row

In [ ]:
import pandas as pd

from src.features.build_features import MODEL_FEATURE_COLUMNS

test_df = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "test.parquet"
)

sample = test_df.iloc[0]

feature_payload = {
    feature: (
        None
        if pd.isna(sample[feature])
        else sample[feature].item()
        if hasattr(sample[feature], "item")
        else sample[feature]
    )
    for feature in MODEL_FEATURE_COLUMNS
}

request_payload = {
    "order_id": str(sample["order_id"]),
    "prediction_timestamp": (
        sample["prediction_timestamp"].isoformat()
    ),
    "features": feature_payload,
}

request_payload

## 7. API smoke test using TestClient

In [ ]:
from fastapi.testclient import TestClient

from app.main import create_app

app = create_app()

with TestClient(app) as client:
    health = client.get("/health")
    model_info = client.get("/model-info")
    prediction = client.post(
        "/predict",
        json=request_payload,
    )
    monitoring = client.get(
        "/monitoring/snapshot"
    )

print("health:")
print(health.json())

print("\nmodel-info:")
print(model_info.json())

print("\nprediction:")
print(prediction.json())

print("\nmonitoring:")
print(monitoring.json())

assert health.status_code == 200
assert prediction.status_code == 200
assert (
    prediction.json()[
        "calibrated_probability"
    ]
    is False
)

## 8. Batch inference

In [ ]:
batch_payload = {
    "items": [
        request_payload,
        {
            **request_payload,
            "order_id": (
                f"{request_payload['order_id']}-copy"
            ),
        },
    ]
}

with TestClient(app) as client:
    batch_response = client.post(
        "/predict/batch",
        json=batch_payload,
    )

batch_response.json()

## 9. Run the service locally

Development server:

```bash
fastapi dev app/main.py
```

Production-style local process:

```bash
uv run uvicorn app.main:app --host 0.0.0.0 --port 8000
```

Swagger:

```text
http://127.0.0.1:8000/docs
```

## 10. Docker

Before the Docker build, make sure the deployment config exists:

```bash
python -m src.serving.build_deployment_config
```

Then:

```bash
docker build -t delivery-delay-api .
docker run --rm -p 8000:8000 delivery-delay-api
```

or:

```bash
docker compose up --build
```

## 11. Monitoring contract

The lightweight runtime snapshot tracks:

- request count;
- prediction count;
- request failures;
- average request latency;
- mean risk score;
- review rate;
- risk-band counts;
- missing feature values.

For a real production platform, export these signals to an external monitoring
system.

Important drift targets from Phase 5 include:

- `promised_delivery_days`;
- score distribution;
- review rate;
- realized positive prevalence;
- future calibration;
- temporal PR-AUC / PR-AUC lift.

## 12. Deployment limitations

The model should not be presented as a fully calibrated probability service.

The production response intentionally includes:

```json
"calibrated_probability": false
```

The API is a portfolio-quality model-serving boundary with reproducible
artifacts, strict input validation, model versioning, containerization and
monitoring hooks.

The next phase should focus on documentation, reproducibility and portfolio
presentation.

# Phase 6 completion criteria

Phase 6 is complete when:

- `models/deployment_config.json` is generated;
- all tests pass;
- `/health` works;
- `/model-info` clearly describes score semantics;
- `/predict` and `/predict/batch` work;
- invalid inputs return validation errors;
- runtime monitoring is observable;
- the Docker image builds;
- the API starts inside the container;
- Swagger documents the inference contract.